# 🏋️‍♂️ Human–Object Interaction (HOI) Multimodal BiLSTM for Sports Action Recognition
## ⚡ Kaggle Notebook Execution Pipeline (82 Sports Categories)

**Thesis Framework**: *Human–Object Interaction-Aware Multimodal CNN–LSTM Framework for Sports Action Recognition*

### 📋 Kaggle Environment Highlights:
- **Hardware**: Free Tesla P100 or 2x T4 GPU (up to 30 hours/week)
- **Input Path**: `/kaggle/input/...` (Read-Only dataset storage)
- **Working Path**: `/kaggle/working/...` (Writable storage for features, checkpoints, and figures)
- **One-Click Download**: Checkpoints and diagrams in `/kaggle/working/` are automatically downloadable from the Kaggle Output pane.

## 1. Verify Kaggle GPU Environment

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Ready: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Kaggle GPU: {torch.cuda.get_device_name(0)}")

## 2. Setup Codebase in `/kaggle/working`

In [ ]:
%cd /kaggle/working
!rm -rf Sport-Action-Detection
!git clone https://github.com/angkon101/Sport-Action-Detection.git
%cd /kaggle/working/Sport-Action-Detection

!pip install -q -r requirements.txt

## 3. Detect & Locate Dataset from Kaggle Inputs
Checks `/kaggle/input` for an attached dataset containing `Model Data` or `ucf action`.

In [ ]:
import os
from pathlib import Path

input_dir = Path("/kaggle/input")
data_dir = None

# Scan for Model Data or ucf action inside /kaggle/input
for p in input_dir.rglob("*"):
    if p.is_dir() and p.name == "Model Data":
        data_dir = p
        break
    elif p.is_dir() and p.name == "ucf action":
        # Populate Model Data split into working directory
        !python data/scripts/populate_model_data_split.py --source-dir "{p}" --target-dir "/kaggle/working/Model Data"
        data_dir = Path("/kaggle/working/Model Data")
        break

if data_dir:
    print(f"[✓] Dataset successfully located at: {data_dir}")
else:
    print("[!] No sports dataset found in /kaggle/input! Please click '+ Add Data' in Kaggle and attach your dataset.")

## 4. Extract Multimodal Features
Extracts Visual (2048-d), Pose (85-d), Object (7-d), and HOI (16-d) features.

In [ ]:
# Extract features to /kaggle/working/data/features
!python run_local.py --extract --data-dir "{data_dir}" --features-dir "/kaggle/working/data/features"

# Pre-flight verification check
!python run_local.py --verify --features-dir "/kaggle/working/data/features"

## 5. Train Model & Generate Outcome Diagrams

In [ ]:
# Train M7 (Full Multimodal + Attention) and generate diagrams
!python run_local.py --train --features-dir "/kaggle/working/data/features" --model-mode M7 --epochs 40 --batch-size 32 --lr 0.0005

## 6. Render the 5 Thesis Outcome Diagrams Inline

In [ ]:
from IPython.display import Image, display
from pathlib import Path

diag_dir = Path("/kaggle/working/Sport-Action-Detection/diagrams/M7")
diagrams = [
    ("1. Training & Validation Loss Curves", diag_dir / "loss_curves.png"),
    ("2. Top-1 & Top-5 Accuracy Curves", diag_dir / "accuracy_curves.png"),
    ("3. 82-Class Confusion Matrix", diag_dir / "confusion_matrix_82.png"),
    ("4. Per-Class Accuracy Rankings", diag_dir / "per_class_accuracy.png"),
    ("5. Temporal Attention Weights Heatmap", diag_dir / "temporal_attention_weights.png"),
]

for title, img_path in diagrams:
    if img_path.exists():
        print(f"\n{'='*70}\n{title}\n{'='*70}")
        display(Image(filename=str(img_path), width=850))
    else:
        print(f"[!] Diagram not found: {img_path}")

## 7. Package Outputs for One-Click Kaggle Download
Copies all diagrams and model checkpoints to `/kaggle/working` root for download.

In [ ]:
!cp -r diagrams/ /kaggle/working/
!cp -r checkpoints/ /kaggle/working/
!zip -r /kaggle/working/Sports_Action_Results.zip /kaggle/working/diagrams /kaggle/working/checkpoints
print("\n[✓] Results packaged! You can download 'Sports_Action_Results.zip' directly from the Kaggle Output pane.")

## 8. Real-Time Action Validation & Live Camera Web App
Launches an interactive web app with a public share link (`https://xxxx.gradio.live`).
- Upload any custom sports video (.mp4, .avi, .mov)
- Record live from your webcam directly in the browser
- Real-time visualization of 17 skeletal joints, sports equipment boxes, and HOI vectors
- Live Top-5 prediction confidence chart

In [ ]:
# Launch Gradio Web App with public shareable link
!python app_web.py --share